# Konaet Cauryvo 202M em µNEX-3F Strict

Este notebook baixa uma revisão imutável, valida assinatura, hashes e Exact 3, carrega `model.mn3f` e gera a resposta na tela. Não chama uma API de texto e não baixa pesos neurais externos.


In [ ]:
%pip install -q "huggingface_hub>=1.0,<2" "cryptography>=46,<47"


In [ ]:
MODEL_ID = "7dsolv/Konaet-Cauryvo-202M-muNEX-3F"
MODEL_REVISION = "b4399cf76835a77bfde0c72dfca6a6048d00f188"
EXPECTED_PAYLOAD_SHA256 = "d78be3243bfb6fd5d0aee46c49ce4faa00df8c5a8913a1f3978edec4b4147d9b"
EXPECTED_PARAMETERS = 202_049_825
EXPECTED_LOGICAL_BITS = EXPECTED_PARAMETERS * 3
print({"modelo": MODEL_ID, "revisao_imutavel": MODEL_REVISION})


In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

model_dir = Path(snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISION))
print("Baixado em:", model_dir)


In [ ]:
import json
import subprocess
import sys

verification = subprocess.run(
    [sys.executable, str(model_dir / "verify_release.py"), str(model_dir)],
    check=True, capture_output=True, text=True, encoding="utf-8",
)
verified = json.loads(verification.stdout)
assert verified["status"] == "PASS"
assert verified["payload_sha256"] == EXPECTED_PAYLOAD_SHA256
assert verified["logical_payload_bits"] == EXPECTED_LOGICAL_BITS
assert verified["logical_bits_per_value"] == 3.0
verified


In [ ]:
import torch

sys.path.insert(0, str(model_dir))
from konaet.foundation.public_inference import load_public_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Carregando model.mn3f em", device)
session = load_public_model(model_dir, device=device)
assert session.trace["payload_sha256"] == EXPECTED_PAYLOAD_SHA256
assert session.trace["logical_bits_per_value"] == 3.0
print({
    "fonte_neural": "model.mn3f",
    "parametros": session.trace["elements"],
    "bits_logicos_por_valor": session.trace["logical_bits_per_value"],
    "criador": session.trace["creator"],
})


In [ ]:
try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = str
    def display(value):
        print(value)

def perguntar(texto, max_novos_tokens=64):
    resultado = session.answer(texto, max_new_tokens=max_novos_tokens)
    display(Markdown(f"**Pessoa:** {texto}\n\n**Konaet:** {resultado['response']}"))
    return resultado

resultado_demo = perguntar("Apresente-se.", max_novos_tokens=64)
assert resultado_demo["source"] == "model.mn3f"


## Converse com o modelo

Escreva abaixo e clique em **Responder**. A execução ocorre dentro desta sessão do Colab.


In [ ]:
import ipywidgets as widgets

entrada = widgets.Textarea(
    value="Explique em uma frase o que é memória causal.",
    description="Pessoa:", layout=widgets.Layout(width="90%", height="90px"),
)
botao = widgets.Button(description="Responder", button_style="success")
saida = widgets.Output()

def ao_clicar(_):
    with saida:
        saida.clear_output(wait=True)
        perguntar(entrada.value, max_novos_tokens=64)

botao.on_click(ao_clicar)
display(entrada, botao, saida)


## Identidade do artefato

Criação e implementação: **Adilson Oliveira / Konaet (7dsolv)**.  
Licença: `LicenseRef-Konaet-ARCL-1.1`.  
Modelo: [7dsolv/Konaet-Cauryvo-202M-muNEX-3F](https://huggingface.co/7dsolv/Konaet-Cauryvo-202M-muNEX-3F/tree/b4399cf76835a77bfde0c72dfca6a6048d00f188).  
Revisão imutável: `b4399cf76835a77bfde0c72dfca6a6048d00f188`.
